In [5]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src.models.content import build_content_features

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
features, movie_ids, vocab_genres = build_content_features(use_tags=False)

print("shape:", features.shape)
print("vocab size:", features.shape[1])
print("all-zero rows:", (features.getnnz(axis=1) == 0).sum())
print("movie_ids length:", len(movie_ids))

In [ ]:
features_full, movie_ids, vocab = build_content_features(use_tags=True)

print("shape:", features_full.shape)
print("all-zero rows:", (features_full.getnnz(axis=1) == 0).sum())

In [12]:
import numpy as np
from sklearn.preprocessing import normalize
from src.data import loaders

idx = {m: i for i, m in enumerate(movie_ids)}
movies = loaders.load_movies()
title_by_id = dict(zip(movies["movieId"], movies["title"]))

X = normalize(features_full)   # L2 normalize so dot product = cosine

def neighbors(movie_id, n=10):
    v = X[idx[movie_id]]
    sims = (X @ v.T).toarray().ravel()
    top = np.argsort(-sims)[1:n+1]
    return [(title_by_id[movie_ids[i]], round(float(sims[i]), 3)) for i in top]

for t, s in neighbors(318):    # Shawshank Redemption
    print(f"{s:.3f}  {t}")

0.498  Seven (a.k.a. Se7en) (1995)
0.422  Lucky Number Slevin (2006)
0.400  Year of the Tiger, The (Año del tigre, El) (2011)
0.397  Sixth Sense, The (1999)
0.396  Jailbreak (1936)
0.382  Magic of Belle Isle, The (2012)
0.377  The Unspoken Truth (1995)
0.377  Unchained (1955)
0.377  Within These Walls (1945)
0.373  Escape from San Quentin (1957)


In [ ]:
from src.models.content import user_profile

train_pos = loaders.positives(loaders.load_split("train"))
history_by_user = train_pos.groupby("userId")["movieId"].apply(set)

for uid in [1, 2, 3]:
    h = history_by_user.loc[uid]
    v = user_profile(X, idx, h)
    top = np.argsort(-v)[:12]
    print(f"user {uid}  ({len(h)} liked movies)")
    print("   ", ", ".join(vocab[i] for i in top))
    print()
